<div class="alert alert-block" style="border: 2px solid #1976D2;background-color:#E3F2FD;padding:5px;font-size:0.9em;">
본 자료는 저작권법 제25조 2항에 의해 보호를 받습니다. 본 자료를 외부에 공개하지 말아주세요.<br>
본 강의만 잘 정리하면, 데이터 분석과 데이터 과학(머신러닝, 인공지능) 모두 가능합니다!<br>
<mb><a href="https://school.fun-coding.org/">잔재미코딩</a> 에서 본 강의 기반 최적화된 로드맵도 확인하실 수 있습니다</b></div>

## 코로나 국가별 확진자 수 추이 분석/시각화

### 날짜별 확진자수 최종 데이터프레임 읽기 (df_confirmed)

In [9]:
import pandas as pd
df_confirmed = pd.read_csv("COVID-19-master/final_df.csv")
df_confirmed.head()

,Country_Region,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,1/30/2020,...,2/28/2023,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023
0,Afghanistan,0,0,0,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,Albania,0,0,0,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,Algeria,0,0,0,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,Andorra,0,0,0,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,Angola,0,0,0,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


In [10]:
df_confirmed.shape

(201, 1144)

### 국가명과 iso2 매칭 테이블 읽기 (country_info)
- 국기 이미지를 가져오기가 쉽지 않음 (1년에 한번씩 변경하였음)
  - 두글자의 ISO 국가 코드 기반(예: KR), 다음 이미지가 현재 가능함 (이후 또 변경시, 본 주피터 노트북에 업데이트 예정)
  - 예: https://public.flourish.studio/country-flags/svg/kr.svg

#### 이슈
  - csv 파일에서 다음 데이터값은 데이터프레임으로 읽어들일 때, 자동으로 결측치로 변환됨
    - ‘’, ‘#N/A’, ‘#N/A N/A’, ‘#NA’, ‘-1.#IND’, ‘-1.#QNAN’, ‘-NaN’, ‘-nan’, ‘1.#IND’, ‘1.#QNAN’, ‘<NA>’, ‘N/A’, ‘NA’, ‘NULL’, ‘NaN’, ‘n/a’, ‘nan’, ‘null’
    - Namibia 국가의 iso2 값이 NA 이므로 결측치 변환을 막기 위해 다음과 같은 옵션 설정 필요
      - keep_default_na=False : 위 데이터값을 자동으로 결측치로 변환하지 않고, na_values로 지정한 데이터만 결측치로 변환
      - na_values='' : 결측치로 변환할 값을 지정

In [11]:
country_info = pd.read_csv("COVID-19-master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv", encoding='utf-8-sig', keep_default_na=False, na_values='')
country_info.head()

,Unnamed: 0.1,Unnamed: 0,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,Combined_Key
0,0,0,NaN,BW,NaN,NaN,NaN,NaN,NaN,Botswana,NaN,NaN,Botswana
1,1,1,NaN,BI,NaN,NaN,NaN,NaN,NaN,Burundi,NaN,NaN,Burundi
2,2,2,NaN,SL,NaN,NaN,NaN,NaN,NaN,Sierra Leone,NaN,NaN,Sierra Leone
3,3,3,4.0,AF,AFG,4.0,NaN,NaN,NaN,Afghanistan,33.93911,67.709953,Afghanistan
4,4,4,8.0,AL,ALB,8.0,NaN,NaN,NaN,Albania,41.15330,20.168300,Albania


In [12]:
country_info[country_info['Country_Region'] == 'Namibia']

,Unnamed: 0.1,Unnamed: 0,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,Combined_Key
115,115,115,516.0,NA,NAM,516.0,NaN,NaN,NaN,Namibia,-22.9576,18.4904,Namibia


In [13]:
country_info = country_info[['iso2', 'Country_Region']]
country_info.head()

,iso2,Country_Region
0,BW,Botswana
1,BI,Burundi
2,SL,Sierra Leone
3,AF,Afghanistan
4,AL,Albania


- 중복 행 제거

In [14]:
country_info.shape

(3560, 2)

In [18]:
# 2026.02.02 업데이트
# 기존 코드
# country_info = country_info.drop_duplicates(subset='Country_Region', keep='last')
# country_info.shape
# 개선 코드: keep='last' 사용 시 일부 국가(UK, Netherlands, Denmark 등)의 iso2가 해외 영토 코드로 매칭되는 문제 발견
# 예: United Kingdom -> TC(터크스 케이커스), Netherlands -> SX(신트마르턴), Denmark -> GL(그린란드)
# keep='first'로 변경하면 대부분 본토 코드가 선택되며, US만 AS(아메리칸 사모아)로 매칭되므로 별도 수정
country_info = country_info.drop_duplicates(subset='Country_Region', keep='first')
country_info.loc[country_info['Country_Region'] == 'US', 'iso2'] = 'US'
country_info.shape

(180, 2)

### 날짜별 국가별 확진자수와 국가별 iso2 값 병합

In [19]:
doc_final_country = pd.merge(df_confirmed, country_info, how='left', on='Country_Region')
doc_final_country.head()

,Country_Region,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,1/30/2020,...,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023,iso2
0,Afghanistan,0,0,0,0,0,0,0,0,0,...,209340,209358,209362,209369,209390,209406,209436,209451,209451,AF
1,Albania,0,0,0,0,0,0,0,0,0,...,334408,334408,334427,334427,334427,334427,334427,334443,334457,AL
2,Algeria,0,0,0,0,0,0,0,0,0,...,271448,271463,271469,271469,271477,271477,271490,271494,271496,DZ
3,Andorra,0,0,0,0,0,0,0,0,0,...,47875,47875,47875,47875,47875,47875,47875,47890,47890,AD
4,Angola,0,0,0,0,0,0,0,0,0,...,105277,105277,105277,105277,105277,105277,105277,105288,105288,AO


In [20]:
doc_final_country.shape

(201, 1145)

#### 없는 데이터(NaN) 확인하기

In [21]:
doc_final_country.isnull().sum()

Country_Region     0
1/22/2020          0
1/23/2020          0
1/24/2020          0
1/25/2020          0
                  ..
3/06/2023          0
3/07/2023          0
3/08/2023          0
3/09/2023          0
iso2              23
Length: 1145, dtype: int64

### 특정 컬럼에 없는 데이터(NaN)가 있는 경우, 해당 행들만 보여주기
- 특정 컬럼 조건에 맞는 행들만 추출 가능
  - dataframe[dataframe[컬럼명] 조건]

In [22]:
doc_final_country[doc_final_country['iso2'].isnull()]

,Country_Region,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,1/30/2020,...,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023,iso2
5,Antarctica,0,0,0,0,0,0,0,0,0,...,11,11,11,11,11,11,11,11,11,NaN
39,Comoros,0,0,0,0,0,0,0,0,0,...,9001,9001,9001,9001,9001,9001,9001,9008,9008,NaN
45,Cruise Ship,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
50,Diamond Princess,0,0,0,0,0,0,0,0,0,...,712,712,712,712,712,712,712,712,712,NaN
93,Kiribati,0,0,0,0,0,0,0,0,0,...,5013,5013,5013,5013,5013,5013,5013,5014,5014,NaN
94,"Korea, North",0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,NaN
102,Lesotho,0,0,0,0,0,0,0,0,0,...,34790,34790,34790,34790,34790,34790,34790,34790,34790,NaN
108,MS Zaandam,0,0,0,0,0,0,0,0,0,...,9,9,9,9,9,9,9,9,9,NaN
114,Marshall Islands,0,0,0,0,0,0,0,0,0,...,15618,15618,15618,15618,15618,15618,15618,15649,15649,NaN
118,Micronesia,0,0,0,0,0,0,0,0,0,...,23948,23948,23948,23948,23948,23948,23948,23948,23948,NaN


In [23]:
doc_final_country[doc_final_country['iso2'].isnull()]['Country_Region']

5                 Antarctica
39                   Comoros
45               Cruise Ship
50          Diamond Princess
93                  Kiribati
94              Korea, North
102                  Lesotho
108               MS Zaandam
114         Marshall Islands
118               Micronesia
126                    Nauru
136                   Others
138                    Palau
153                    Samoa
155    Sao Tome and Principe
164          Solomon Islands
170     Summer Olympics 2020
176               Tajikistan
181                    Tonga
185                   Tuvalu
193                  Vanuatu
197     Winter Olympics 2022
198                    Yemen
Name: Country_Region, dtype: object

| 번호 | 항목                    | 설명                           |
|------|-------------------------|--------------------------------|
| 5    | Antarctica              | 국가가 아닌 남극 대륙            |
| 45   | Cruise Ship             | 국가가 아닌 유람선 관련 데이터   |
| 50   | Diamond Princess        | 국가가 아닌 유람선 (유명 크루즈 선박)|
| 108  | MS Zaandam              | 국가가 아닌 유람선 (크루즈 선박)   |
| 136  | Others                  | 국가 외 기타 항목               |
| 170  | Summer Olympics 2020    | 국가가 아닌 이벤트 (올림픽 대회)  |
| 197  | Winter Olympics 2022    | 국가가 아닌 이벤트 (올림픽 대회)  |


#### 참고: 복합 조건은 
- dataframe[(조건1) & (조건2)] : 조건1과 조건2 모두 만족 (and 조건)
- dataframe[(조건1) | (조건2)] : 조건1 또는 조건2 만족 (or 조건)
- 조건에 괄호를 넣는 것이 오동작을 방지하기 위해 좋음

```
nan_rows = doc_final_country[(doc_final_country['iso2'].isnull()) & (doc_final_country['4/01/2020'] == 0)]
nan_rows.head()
```

### 특정 컬럼에 없는 데이터 삭제하기

In [24]:
doc_final_country = doc_final_country.dropna(subset=['iso2'])

In [25]:
doc_final_country[doc_final_country['iso2'].isnull()]

,Country_Region,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,1/30/2020,...,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023,iso2


### 국기 링크를 기존 컬럼 기반해서 만들어, 데이터프레임에 붙이기
- 국기 이미지를 가져오기가 쉽지 않음 (1년에 한번씩 변경하였음)
  - 두글자의 ISO 국가 코드 기반(예: KR), 다음 이미지가 현재 가능함 (이후 또 변경시, 본 주피터 노트북에 업데이트 예정)
  - 예: https://public.flourish.studio/country-flags/svg/kr.svg

- 국기 링크 만들기

In [26]:
iso2= 'AD'
flag_link = 'https://public.flourish.studio/country-flags/svg/' + iso2.lower() + '.svg'
flag_link

'https://public.flourish.studio/country-flags/svg/ad.svg'

#### 참고: 데이터프레임에 신규 컬럼 추가하기
- 데이터 프레임에 신규 컬럼 추가하기
  - 데이터프레임[신규컬럼] = pd.Series(data=데이터)
    - 데이터는 행에 맞게, [0, 1, 2] 와 같은 형태로 넣을 수 있음
- 기존 데이터프레임 컬럼 값을 기반으로 신규 컬럼 추가하기
  - apply() 함수를 사용

In [27]:
def create_flag_link(column):
    flag_link = 'https://public.flourish.studio/country-flags/svg/' + column.lower() + '.svg'
    print(flag_link)
    return flag_link

doc_final_country['iso2'] = doc_final_country['iso2'].apply(create_flag_link)

https://public.flourish.studio/country-flags/svg/af.svg
https://public.flourish.studio/country-flags/svg/al.svg
https://public.flourish.studio/country-flags/svg/dz.svg
https://public.flourish.studio/country-flags/svg/ad.svg
https://public.flourish.studio/country-flags/svg/ao.svg
https://public.flourish.studio/country-flags/svg/ag.svg
https://public.flourish.studio/country-flags/svg/ar.svg
https://public.flourish.studio/country-flags/svg/am.svg
https://public.flourish.studio/country-flags/svg/au.svg
https://public.flourish.studio/country-flags/svg/at.svg
https://public.flourish.studio/country-flags/svg/az.svg
https://public.flourish.studio/country-flags/svg/bs.svg
https://public.flourish.studio/country-flags/svg/bh.svg
https://public.flourish.studio/country-flags/svg/bd.svg
https://public.flourish.studio/country-flags/svg/bb.svg
https://public.flourish.studio/country-flags/svg/by.svg
https://public.flourish.studio/country-flags/svg/be.svg
https://public.flourish.studio/country-flags/svg

### 데이터프레임 컬럼 조정하기

In [28]:
doc_final_country.head()

,Country_Region,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,1/30/2020,...,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023,iso2
0,Afghanistan,0,0,0,0,0,0,0,0,0,...,209340,209358,209362,209369,209390,209406,209436,209451,209451,https://public.flourish.studio/country-flags/s...
1,Albania,0,0,0,0,0,0,0,0,0,...,334408,334408,334427,334427,334427,334427,334427,334443,334457,https://public.flourish.studio/country-flags/s...
2,Algeria,0,0,0,0,0,0,0,0,0,...,271448,271463,271469,271469,271477,271477,271490,271494,271496,https://public.flourish.studio/country-flags/s...
3,Andorra,0,0,0,0,0,0,0,0,0,...,47875,47875,47875,47875,47875,47875,47875,47890,47890,https://public.flourish.studio/country-flags/s...
4,Angola,0,0,0,0,0,0,0,0,0,...,105277,105277,105277,105277,105277,105277,105277,105288,105288,https://public.flourish.studio/country-flags/s...


#### 1. 데이터프레임에서 필요한 컬럼만 선택하기
- 데이터프레임.columns.tolist() : 컬럼명을 리스트로 변환

<img src="https://davelee-fun.github.io/fixeddata/covid_data_format.png">

In [29]:
cols = doc_final_country.columns.tolist()
cols.remove('iso2')
cols.insert(1, 'iso2')
cols

['Country_Region',
 'iso2',
 '1/22/2020',
 '1/23/2020',
 '1/24/2020',
 '1/25/2020',
 '1/26/2020',
 '1/27/2020',
 '1/28/2020',
 '1/29/2020',
 '1/30/2020',
 '1/31/2020',
 '2/01/2020',
 '2/02/2020',
 '2/03/2020',
 '2/04/2020',
 '2/05/2020',
 '2/06/2020',
 '2/07/2020',
 '2/08/2020',
 '2/09/2020',
 '2/10/2020',
 '2/11/2020',
 '2/12/2020',
 '2/13/2020',
 '2/14/2020',
 '2/15/2020',
 '2/16/2020',
 '2/17/2020',
 '2/18/2020',
 '2/19/2020',
 '2/20/2020',
 '2/21/2020',
 '2/22/2020',
 '2/23/2020',
 '2/24/2020',
 '2/25/2020',
 '2/26/2020',
 '2/27/2020',
 '2/28/2020',
 '2/29/2020',
 '3/01/2020',
 '3/02/2020',
 '3/03/2020',
 '3/04/2020',
 '3/05/2020',
 '3/06/2020',
 '3/07/2020',
 '3/08/2020',
 '3/09/2020',
 '3/10/2020',
 '3/11/2020',
 '3/12/2020',
 '3/13/2020',
 '3/14/2020',
 '3/15/2020',
 '3/16/2020',
 '3/17/2020',
 '3/18/2020',
 '3/19/2020',
 '3/20/2020',
 '3/21/2020',
 '3/22/2020',
 '3/23/2020',
 '3/24/2020',
 '3/25/2020',
 '3/26/2020',
 '3/27/2020',
 '3/28/2020',
 '3/29/2020',
 '3/30/2020',
 '3/31

#### 내가 정한 조건대로 정렬하기
- **`sorted(..., key=lambda x: pd.to_datetime(x, format='%m/%d/%Y'))`**:  
  - `sorted` 함수는 리스트를 정렬하여 새 리스트를 반환합니다.
  - **`key=lambda x: ...`**:  
    - 각 요소 `x`에 대해 정렬 기준을 지정합니다.
    - `x`는 날짜 형식의 문자열일 것으로 가정됩니다.
  - **`pd.to_datetime(x, format='%m/%d/%Y')`**:  
    - `x`를 지정된 날짜 포맷(`월/일/연도`)에 따라 datetime 객체로 변환합니다.
    - 변환된 datetime 객체를 기준으로 정렬되므로, 날짜 순서대로 정렬됩니다.

In [30]:
sorted(cols[2:], key=lambda x: pd.to_datetime(x, format='%m/%d/%Y'))

['1/22/2020',
 '1/23/2020',
 '1/24/2020',
 '1/25/2020',
 '1/26/2020',
 '1/27/2020',
 '1/28/2020',
 '1/29/2020',
 '1/30/2020',
 '1/31/2020',
 '2/01/2020',
 '2/02/2020',
 '2/03/2020',
 '2/04/2020',
 '2/05/2020',
 '2/06/2020',
 '2/07/2020',
 '2/08/2020',
 '2/09/2020',
 '2/10/2020',
 '2/11/2020',
 '2/12/2020',
 '2/13/2020',
 '2/14/2020',
 '2/15/2020',
 '2/16/2020',
 '2/17/2020',
 '2/18/2020',
 '2/19/2020',
 '2/20/2020',
 '2/21/2020',
 '2/22/2020',
 '2/23/2020',
 '2/24/2020',
 '2/25/2020',
 '2/26/2020',
 '2/27/2020',
 '2/28/2020',
 '2/29/2020',
 '3/01/2020',
 '3/02/2020',
 '3/03/2020',
 '3/04/2020',
 '3/05/2020',
 '3/06/2020',
 '3/07/2020',
 '3/08/2020',
 '3/09/2020',
 '3/10/2020',
 '3/11/2020',
 '3/12/2020',
 '3/13/2020',
 '3/14/2020',
 '3/15/2020',
 '3/16/2020',
 '3/17/2020',
 '3/18/2020',
 '3/19/2020',
 '3/20/2020',
 '3/21/2020',
 '3/22/2020',
 '3/23/2020',
 '3/24/2020',
 '3/25/2020',
 '3/26/2020',
 '3/27/2020',
 '3/28/2020',
 '3/29/2020',
 '3/30/2020',
 '3/31/2020',
 '4/01/2020',
 '4/02

#### 참고: 오름차순 정렬과 내림차순 정렬
- **오름차순 정렬 (낮은 순)**  
  기본적으로 `sorted` 함수는 오름차순 정렬(가장 작은 값부터 큰 값 순)을 수행합니다. 날짜의 경우, 이는 **이른 날짜부터 늦은 날짜** 순으로 정렬됨을 의미합니다.
  
  예시:
  ```python
  # 이른 날짜(낮은 값) -> 늦은 날짜(높은 값) 순
  sorted_list = sorted(cols[2:], key=lambda x: pd.to_datetime(x, format='%m/%d/%Y'))
  ```

- **내림차순 정렬 (높은 순)**  
  내림차순 정렬(큰 값부터 작은 값 순)을 원한다면, `sorted` 함수의 `reverse` 매개변수를 `True`로 설정하면 됩니다. 날짜의 경우, 이는 **늦은 날짜부터 이른 날짜** 순으로 정렬됨을 의미합니다.
  
  예시:
  ```python
  # 늦은 날짜(높은 값) -> 이른 날짜(낮은 값) 순
  sorted_list_desc = sorted(cols[2:], key=lambda x: pd.to_datetime(x, format='%m/%d/%Y'), reverse=True)
  ```

이처럼 `reverse` 매개변수를 통해 날짜 정렬의 순서를 쉽게 조절할 수 있습니다.  
- `reverse=False` 또는 생략 시: 오름차순 (낮은 순)  
- `reverse=True` 시: 내림차순 (높은 순)

#### 리스트 합치기
- 리스트타입 + 리스트타입 으로 리스트를 합칠 수 있음 

In [31]:
cols = cols[:2] + sorted(cols[2:], key=lambda x: pd.to_datetime(x, format='%m/%d/%Y'))
cols

['Country_Region',
 'iso2',
 '1/22/2020',
 '1/23/2020',
 '1/24/2020',
 '1/25/2020',
 '1/26/2020',
 '1/27/2020',
 '1/28/2020',
 '1/29/2020',
 '1/30/2020',
 '1/31/2020',
 '2/01/2020',
 '2/02/2020',
 '2/03/2020',
 '2/04/2020',
 '2/05/2020',
 '2/06/2020',
 '2/07/2020',
 '2/08/2020',
 '2/09/2020',
 '2/10/2020',
 '2/11/2020',
 '2/12/2020',
 '2/13/2020',
 '2/14/2020',
 '2/15/2020',
 '2/16/2020',
 '2/17/2020',
 '2/18/2020',
 '2/19/2020',
 '2/20/2020',
 '2/21/2020',
 '2/22/2020',
 '2/23/2020',
 '2/24/2020',
 '2/25/2020',
 '2/26/2020',
 '2/27/2020',
 '2/28/2020',
 '2/29/2020',
 '3/01/2020',
 '3/02/2020',
 '3/03/2020',
 '3/04/2020',
 '3/05/2020',
 '3/06/2020',
 '3/07/2020',
 '3/08/2020',
 '3/09/2020',
 '3/10/2020',
 '3/11/2020',
 '3/12/2020',
 '3/13/2020',
 '3/14/2020',
 '3/15/2020',
 '3/16/2020',
 '3/17/2020',
 '3/18/2020',
 '3/19/2020',
 '3/20/2020',
 '3/21/2020',
 '3/22/2020',
 '3/23/2020',
 '3/24/2020',
 '3/25/2020',
 '3/26/2020',
 '3/27/2020',
 '3/28/2020',
 '3/29/2020',
 '3/30/2020',
 '3/31

#### 데이터프레임 컬럼 위치 변경

In [32]:
doc_final_country = doc_final_country[cols]

In [33]:
doc_final_country.head()

,Country_Region,iso2,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,...,2/28/2023,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023
0,Afghanistan,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,Albania,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,Algeria,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,Andorra,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,Angola,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


#### 컬럼명 변경

In [34]:
cols[1] = 'Country_Flag'

In [35]:
doc_final_country.columns = cols
doc_final_country.head()

,Country_Region,Country_Flag,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,...,2/28/2023,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023
0,Afghanistan,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,Albania,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,Algeria,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,Andorra,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,Angola,https://public.flourish.studio/country-flags/s...,0,0,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


### 최종 가공 완료 파일 저장

In [36]:
doc_final_country.to_csv("COVID-19-master/final_covid_data_for_graph.csv")

### 그래프 만들기
- https://app.flourish.studio/ 로그인
- New visualization 선택 -> Bar Chart Race 선택 -> 데이터 수정 (파일 업로드)
- 최종 그래프 예: https://public.flourish.studio/visualisation/17998107/

<div class="alert alert-block" style="border: 2px solid #1976D2;background-color:#E3F2FD;padding:5px;font-size:0.9em;">
본 자료는 저작권법 제25조 2항에 의해 보호를 받습니다. 본 자료를 외부에 공개하지 말아주세요.<br>
본 강의만 잘 정리하면, 데이터 분석과 데이터 과학(머신러닝, 인공지능) 모두 가능합니다!<br>
<b><a href="https://school.fun-coding.org/">잔재미코딩</a> 에서 본 강의 기반 최적화된 로드맵도 확인하실 수 있습니다</b></div>